# dLEM full-chromosome analysis: H1-hESC chr10 + CTCF

This notebook fits dLEM to a full human chromosome and compares the inferred
cohesin loading/unloading rates with CTCF ChIP-seq occupancy.

**Cell line**: H1-hESC (ENCODE)  
**Hi-C data**: `4DNFI9GMP2J8` (4D Nucleome, 10 kb resolution)  
**CTCF ChIP-seq**: ENCODE accession `ENCFF269OPL` (H1-hESC, narrowPeak)  
**Chromosome**: chr10 (~133 Mb, ~13,300 bins at 10 kb)

**Data downloads** (run cell 2 once): the Hi-C file is ~13 GB and the CTCF
file is ~5 MB. Both are cached in `data/` and skipped on subsequent runs.

**Prerequisites**: install dLEM with `pip install dlem-jax`, then run this
notebook from the `docs/` directory or with `jupyter execute docs/full_analysis.ipynb`.

In [ ]:
import os
from pathlib import Path
import urllib.request
import numpy as np
import matplotlib
matplotlib.use('Agg')          # remove this line for interactive Jupyter
import matplotlib.pyplot as plt
import seaborn as sns
import jax.numpy as jnp
import cooler
import pandas as pd

from dlem.api import fetch_band, train_dlem, fit_band_row_profile_sliding, flip_diag_row
from dlem.core import normalize_expected_observed, jax_forward_generate

## 1. Download data

The Hi-C `.mcool` file (~13 GB) is downloaded from the 4D Nucleome public S3 bucket.
The CTCF narrowPeak file (~5 MB) is downloaded from the ENCODE portal.
Both are skipped on subsequent runs if the files already exist.

In [ ]:
DATA_DIR  = Path('data')           # relative to docs/ — set by jupyter execute automatically
MCOOL     = DATA_DIR / '4DNFI9GMP2J8.mcool'
CTCF_FILE = DATA_DIR / 'ENCFF269OPL.bed.gz'

MCOOL_URL = (
    'https://4dn-open-data-public.s3.amazonaws.com/fourfront-webprod/wfoutput/'
    'd13aa1ea-053c-4113-94fe-a1e7ab1dbbab/4DNFI9GMP2J8.mcool'
)
CTCF_URL = (
    'https://www.encodeproject.org/files/ENCFF269OPL/@@download/ENCFF269OPL.bed.gz'
)


def _download(url, dest, chunk=1 << 20):
    dest = Path(dest)
    if dest.exists():
        print(f'Already exists: {dest.name}  ({dest.stat().st_size / 1e9:.2f} GB)')
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.tmp')
    print(f'Downloading {dest.name} ...')
    try:
        with urllib.request.urlopen(url) as resp, open(tmp, 'wb') as fh:
            total = int(resp.headers.get('Content-Length', 0))
            done  = 0
            while True:
                block = resp.read(chunk)
                if not block:
                    break
                fh.write(block)
                done += len(block)
                if total:
                    print(f'\r  {done / total * 100:5.1f}%  '
                          f'({done / 1e9:.2f} / {total / 1e9:.2f} GB)',
                          end='', flush=True)
        tmp.rename(dest)
        print(f'\nSaved: {dest}  ({dest.stat().st_size / 1e9:.2f} GB)')
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


_download(MCOOL_URL, MCOOL)
_download(CTCF_URL,  CTCF_FILE)

## 2. Load Hi-C data

`fetch_band` reads the contact matrix and applies adaptive coarse-graining and
NaN filling, returning a (n_diagonals × n_bins) band matrix.
Here we use 170 diagonals (= 1.7 Mb at 10 kb) for training, which is sufficient
to capture the relevant contact range.

In [ ]:
RESOLUTION = 10_000

clr       = cooler.Cooler(f'{MCOOL}::resolutions/{RESOLUTION}')
CHR10_LEN = int(clr.chromsizes['chr10'])
N_BINS    = CHR10_LEN // RESOLUTION + 1
REGION    = f'chr10:0-{CHR10_LEN}'

print(f'chr10: {CHR10_LEN / 1e6:.1f} Mb  →  {N_BINS} bins at {RESOLUTION // 1000} kb')

band = fetch_band(str(MCOOL), RESOLUTION, REGION, width=170)   # (170, N_BINS)
band_train = band[1:170, :]                                     # skip row 0
print(f'band shape: {band.shape},  training rows: {band_train.shape}')

## 3. Estimate distance-decay parameter

The `slowdown` parameter (cohesin detachment rate per step) can be estimated
by fitting the genome-average contact-frequency decay profile.
`fit_band_row_profile_sliding` fits a power-law + exponential model across
sliding windows and returns the median exponential rate `d` (negative);
`slowdown ≈ |d|`.

In [ ]:
decay = fit_band_row_profile_sliding(
    band,
    start_row=5,
    extent=160,
    window_size=300,
)
slowdown = float(abs(decay['params']['d']))
print(f'Estimated slowdown: {slowdown:.4f}  (paper value: 0.025)')

## 4. Train dLEM on chr10

In [ ]:
result = train_dlem(
    band_train,
    steps=10,
    start_row=5,
    slowdown=slowdown,
    learning_rate=1e-2,
    train_steps=300,
    loss_type='multinomial',
    weight_power=0,
    auto_stop_metric='mse',
    verbose=True,
)
p_left  = np.array(result['p_left_mse'])
p_right = np.array(result['p_right_mse'])
print(f'p_left / p_right shape: {p_left.shape}  (one value per chr10 bin)')

## 5. Load CTCF peaks

CTCF ChIP-seq peaks are read from the ENCODE narrowPeak file (column 6 = fold-change
signal). Peak signals are summed per 10 kb bin.

In [ ]:
ctcf_raw = pd.read_csv(
    CTCF_FILE, sep='\t', compression='gzip', header=None,
    usecols=[0, 1, 6], names=['chrom', 'start', 'signal'],
)
ctcf_chr10 = ctcf_raw[ctcf_raw['chrom'] == 'chr10'].copy()
ctcf_chr10['bin'] = (ctcf_chr10['start'] // RESOLUTION).astype(int)
ctcf_signal = (
    ctcf_chr10.groupby('bin')['signal']
    .sum()
    .reindex(range(N_BINS), fill_value=0.0)
)
print(f'CTCF peaks on chr10: {len(ctcf_chr10)}  '
      f'(occupied bins: {(ctcf_signal > 0).sum()}, '
      f'max signal: {ctcf_signal.max():.1f})')

## 6. Full-chromosome overview

Three panels with shared x-axis (chr10 position in Mb):
- **L / R**: fitted cohesin loading rates — peaks indicate regions with directed extrusion
- **CTCF signal**: fold-change from ChIP-seq; CTCF-bound sites are expected to co-localise
  with high L or R values (cohesin stalls at CTCF)
- **Barrier proxy** `(1−L)·(1−R)`: high where both L and R are low — marks potential
  cohesin-poor regions or boundaries

In [ ]:
coords_mb = np.arange(N_BINS) * RESOLUTION / 1e6

fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1.5, 1.5], 'hspace': 0.05})

# Row 0 — cohesin speed tracks
axes[0].plot(coords_mb, p_left,  lw=0.6, c='tab:blue',   label='L')
axes[0].plot(coords_mb, p_right, lw=0.6, c='tab:orange', label='R')
axes[0].set_ylim(0, 1.05)
axes[0].set_yticks([0, 0.5, 1])
axes[0].set_ylabel('speed', fontsize=9)
axes[0].legend(fontsize=8, loc='upper right', framealpha=0.5)
axes[0].set_title('H1-hESC chr10  |  dLEM cohesin speed tracks vs CTCF ChIP-seq  '
                   '(10 kb resolution)', fontsize=9)

# Row 1 — CTCF signal
axes[1].fill_between(coords_mb, ctcf_signal.values, color='tab:green', lw=0, alpha=0.8)
axes[1].set_ylabel('CTCF\nsignal', fontsize=9)
axes[1].tick_params(labelsize=8)

# Row 2 — barrier proxy
barrier = (1 - p_left) * (1 - p_right)
axes[2].plot(coords_mb, barrier, lw=0.6, c='tab:purple')
axes[2].set_ylim(0, 1.05)
axes[2].set_yticks([0, 0.5, 1])
axes[2].set_ylabel('barrier\nproxy', fontsize=9)
axes[2].set_xlabel('chr10 position (Mb)', fontsize=9)
axes[2].tick_params(labelsize=8)

for ax in axes:
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.savefig('full_analysis_chr10.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: full_analysis_chr10.png')

## 7. Reference locus: chr10:20.5–22.5 Mb

Zoom into the 2 Mb region used as a reference in the publication.
The split contact map shows observed contacts (upper triangle) and
dLEM predictions (lower triangle); CTCF peak positions are marked
below the contact map.

In [ ]:
PATCH_START = 2050   # chr10:20.5 Mb = 20,500,000 / 10,000
PATCH_SPAN  = 200    # 200 bins × 10 kb = 2 Mb

p_left_patch  = p_left [PATCH_START:PATCH_START + PATCH_SPAN]
p_right_patch = p_right[PATCH_START:PATCH_START + PATCH_SPAN]

pred_band = np.array(jax_forward_generate(
    jnp.array(p_left_patch,  jnp.float32),
    jnp.array(p_right_patch, jnp.float32),
    slowdown,
    PATCH_SPAN,
))

# Re-fetch observed band for this region at full patch depth (200 diagonals)
obs_band = fetch_band(
    str(MCOOL), RESOLUTION,
    f'chr10:{PATCH_START * RESOLUTION}-{(PATCH_START + PATCH_SPAN) * RESOLUTION}',
    width=PATCH_SPAN,
)

def _log_eo(b):
    return np.array(normalize_expected_observed(jnp.asarray(b, jnp.float32)))

pred_sq  = flip_diag_row(_log_eo(pred_band))
obs_sq   = flip_diag_row(_log_eo(obs_band))
combined = np.triu(obs_sq) + np.tril(pred_sq.T, k=-1)

# CTCF peak positions within the patch
ctcf_patch_bins = ctcf_chr10[
    (ctcf_chr10['bin'] >= PATCH_START) &
    (ctcf_chr10['bin'] <  PATCH_START + PATCH_SPAN)
]['bin'].values
ctcf_patch_mb = ctcf_patch_bins * RESOLUTION / 1e6

patch_start_mb = PATCH_START * RESOLUTION / 1e6   # 20.5
patch_end_mb   = patch_start_mb + PATCH_SPAN * RESOLUTION / 1e6  # 22.5
coords_patch   = np.linspace(patch_start_mb, patch_end_mb, PATCH_SPAN, endpoint=False)

cmap = sns.color_palette('vlag', as_cmap=True)

fig = plt.figure(figsize=(7, 7))
gs = fig.add_gridspec(
    3, 2,
    hspace=0, wspace=0,
    height_ratios=[1, 5, 0.4], width_ratios=[5, 1],
)
ax_main  = fig.add_subplot(gs[1, 0])
ax_top   = fig.add_subplot(gs[0, 0], sharex=ax_main)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)
ax_ctcf  = fig.add_subplot(gs[2, 0], sharex=ax_main)

im = ax_main.matshow(
    combined, cmap=cmap, vmin=-2, vmax=2,
    extent=[patch_start_mb, patch_end_mb, patch_end_mb, patch_start_mb],
)
ax_main.xaxis.set_ticks_position('bottom')
ax_main.tick_params(labelsize=8)
ax_main.set_xlabel('chr10 (Mb)', fontsize=9)
ax_main.set_ylabel('chr10 (Mb)', fontsize=9)
ax_main.text(0.97, 0.97, 'obs',  fontsize=8, ha='right', va='top',
             transform=ax_main.transAxes, color='white')
ax_main.text(0.03, 0.03, 'pred', fontsize=8, ha='left',  va='bottom',
             transform=ax_main.transAxes, color='white')

ax_top.plot(coords_patch, p_left_patch, lw=1.5, c='tab:blue')
ax_top.set_ylim(0, 1.05)
ax_top.set_yticks([0, 0.5, 1])
ax_top.set_yticklabels(['', '', '1'], fontsize=7)
ax_top.set_ylabel('L', fontsize=9)
ax_top.xaxis.set_visible(False)
ax_top.set_title('chr10:20.5–22.5 Mb  |  H1-hESC 10 kb', fontsize=9)

ax_right.plot(p_right_patch, coords_patch, lw=1.5, c='tab:orange')
ax_right.set_xlim(0, 1.05)
ax_right.set_xticks([0, 0.5, 1])
ax_right.set_xticklabels(['', '', '1'], fontsize=7)
ax_right.set_xlabel('R', fontsize=9)
ax_right.xaxis.set_ticks_position('top')
ax_right.xaxis.set_label_position('top')
ax_right.yaxis.set_visible(False)

if len(ctcf_patch_mb):
    ax_ctcf.vlines(ctcf_patch_mb, 0, 1, lw=0.8, color='tab:green')
ax_ctcf.set_ylim(0, 1)
ax_ctcf.set_yticks([])
ax_ctcf.set_ylabel('CTCF', fontsize=8, rotation=0, ha='right', va='center')
ax_ctcf.tick_params(labelsize=7)
ax_ctcf.set_xlabel('chr10 (Mb)', fontsize=9)

# Force exact alignment
fig.canvas.draw()
main_bbox  = ax_main.get_position()
top_bbox   = ax_top.get_position()
right_bbox = ax_right.get_position()
ctcf_bbox  = ax_ctcf.get_position()
total_w = right_bbox.x0 + right_bbox.width - main_bbox.x0
right_w = total_w / 6
ax_top.set_position([main_bbox.x0, top_bbox.y0, main_bbox.width, top_bbox.height])
ax_right.set_position([main_bbox.x0 + main_bbox.width, main_bbox.y0,
                        right_w, main_bbox.height])
ax_ctcf.set_position([main_bbox.x0, ctcf_bbox.y0, main_bbox.width, ctcf_bbox.height])

cbar_ax = fig.add_axes([
    main_bbox.x0 + main_bbox.width + right_w + 0.01,
    main_bbox.y0,
    0.025,
    main_bbox.height,
])
fig.colorbar(im, cax=cbar_ax, label='log(obs/exp)')

plt.savefig('full_analysis_locus.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: full_analysis_locus.png')

## 8. Cohesin activity vs CTCF signal

Correlation between per-bin CTCF fold-change and total cohesin speed `L + R`.
Bins with zero CTCF signal are excluded from the scatter.

In [ ]:
from scipy.stats import pearsonr

cohesin_activity = p_left + p_right
ctcf_arr = ctcf_signal.values

mask = ctcf_arr > 0
x = np.log1p(ctcf_arr[mask])
y = cohesin_activity[mask]
r, pval = pearsonr(x, y)

fig, ax = plt.subplots(figsize=(5, 4))
ax.hexbin(x, y, gridsize=60, cmap='Blues', mincnt=1)
ax.set_xlabel('log(1 + CTCF signal)', fontsize=9)
ax.set_ylabel('L + R  (cohesin speed)', fontsize=9)
ax.set_title(f'chr10 bins with CTCF peaks (n={mask.sum():,})  |  Pearson r = {r:.3f}',
             fontsize=9)
ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('full_analysis_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Pearson r = {r:.3f}  (p = {pval:.2e})')
print('Saved: full_analysis_scatter.png')